In [1]:
# Cell 1: Database Connection
import os
from sqlalchemy import create_engine, text

# --- CONFIGURATION ---
DB_USER = "postgres"
DB_PASS = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "postgres"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL)
    # Test the connection
    with engine.connect() as conn:
        res = conn.execute(text("SELECT 1")).scalar()
        print(f"✅ Database Connection Successful! (Test result: {res})")
except Exception as e:
    print(f"❌ Connection Failed: {e}")

✅ Database Connection Successful! (Test result: 1)


In [ ]:
import time
import datetime
import pandas as pd
import numpy as np
import pyotp
import requests
import json
from SmartApi import SmartConnect
from tqdm.notebook import tqdm # For the progress bar
import logging

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger()

# ==========================================
# 1. MULTI-KEY CONFIGURATION
# ==========================================
# Add your 5 API Keys here. 
# NOTE: The CLIENT_CODE and PASSWORD usually remain the same, 
# but if you have 5 different accounts, change those too.
CREDENTIALS_POOL = [
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
]

class AngelKeyManager:
    def __init__(self, creds_list):
        self.sessions = []
        self.current_index = 0
        print(f"Initializing {len(creds_list)} API Sessions...")
        
        for cred in creds_list:
            try:
                obj = SmartConnect(api_key=cred['api_key'])
                totp = pyotp.TOTP(cred['totp']).now()
                data = obj.generateSession(cred['client'], cred['pwd'], totp)
                if data['status']:
                    self.sessions.append(obj)
                    print(f" >> Session Active for Key: ...{cred['api_key'][-4:]}")
                else:
                    print(f" >> Failed for Key: ...{cred['api_key'][-4:]} | {data['message']}")
            except Exception as e:
                print(f" >> Error: {e}")
        
        if not self.sessions:
            raise Exception("No API Sessions could be established!")

    def get_session(self):
        """Returns a session and rotates the index"""
        session = self.sessions[self.current_index]
        # Rotate to next key
        self.current_index = (self.current_index + 1) % len(self.sessions)
        return session

# Initialize the Pool (Run this once)
key_manager = AngelKeyManager(CREDENTIALS_POOL)
print("Setup Complete. Uncomment the initialization line above after filling credentials.")